<a href="https://colab.research.google.com/github/rdolgov/lerobot-training/blob/main/smolvla_google_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LeRobot ACT training on Google Colab

This notebook trains an ACT policy on your SO-101 dataset from Hugging Face Hub, logs to W&B, and pushes the trained policy back to Hugging Face Hub.

Before running it in Colab, add these secrets in the Colab Secrets panel:

- `HF_TOKEN`: a Hugging Face token with read access to the dataset and write access to the policy repo
- `WANDB_API_KEY`: your Weights & Biases API key

Use a GPU runtime. Current Colab Python 3.12 runtimes work with this repo, so the notebook uses the Colab kernel directly instead of creating a separate venv or Conda env.


In [1]:
# @title 1. Configure the run
from datetime import datetime
from pathlib import Path

HF_USER = "romando"  # @param {type:"string"}
DATASET_REPO_ID = "romando/record-test-may16-1200_20260516_124636"  # @param {type:"string"}
POLICY_REPO_NAME = "f\"smolvla-so101-{DATASET_REPO_ID}\""  # @param {type:"string"}
POLICY_PRIVATE = True  # @param {type:"boolean"}

WANDB_PROJECT = "lerobot"  # @param {type:"string"}
WANDB_ENTITY = "roman-dolgov-rd"  # @param {type:"string"}
ENABLE_WANDB = True  # @param {type:"boolean"}

TRAIN_STEPS = 100  # @param {type:"integer"}
BATCH_SIZE = 8  # @param {type:"integer"}
NUM_WORKERS = 8  # @param {type:"integer"}
LOG_FREQ = 50  # @param {type:"integer"}
EVAL_FREQ = 0  # @param {type:"integer"}
USE_AMP = True  # @param {type:"boolean"}

LEROBOT_REPO_URL = "https://github.com/huggingface/lerobot.git"  # @param {type:"string"}
LEROBOT_REF = "main"  # @param {type:"string"}
LEROBOT_DIR = Path("/content/lerobot")

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
JOB_NAME = f"job_{POLICY_REPO_NAME}_{RUN_STAMP}"
OUTPUT_DIR = LEROBOT_DIR / "outputs" / "train" / JOB_NAME
POLICY_REPO_ID = f"{HF_USER}/{POLICY_REPO_NAME}"
SAVE_FREQ = max(1000, TRAIN_STEPS // 5)

print(f"Dataset: {DATASET_REPO_ID}")
print(f"Policy repo: {POLICY_REPO_ID}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Job name: {JOB_NAME}")


Dataset: romando/record-test-may16-1200_20260516_124636
Policy repo: romando/f"smolvla-so101-{DATASET_REPO_ID}"
Output dir: /content/lerobot/outputs/train/job_f"smolvla-so101-{DATASET_REPO_ID}"_20260517_231123
Job name: job_f"smolvla-so101-{DATASET_REPO_ID}"_20260517_231123


In [2]:
# @title 2. Check Python and install system packages
import subprocess
import sys

print(sys.version)
if sys.version_info < (3, 12):
    raise RuntimeError(
        "This LeRobot checkout requires Python 3.12+. In Colab, select a current Python 3 runtime."
    )

subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
subprocess.run(["sudo", "apt-get", "install", "-y", "-qq", "ffmpeg", "git", "git-lfs"], check=True)
subprocess.run(["git", "lfs", "install", "--skip-smudge"], check=True)


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


CompletedProcess(args=['git', 'lfs', 'install', '--skip-smudge'], returncode=0)

In [ ]:
# @title 3. Clone LeRobot and install training dependencies
import os
import subprocess
import sys
from pathlib import Path

LEROBOT_REPO_URL = globals().get("LEROBOT_REPO_URL", "https://github.com/huggingface/lerobot.git")
LEROBOT_REF = globals().get("LEROBOT_REF", "main")
LEROBOT_DIR = globals().get("LEROBOT_DIR", Path("/content/lerobot"))

env = os.environ.copy()
env["GIT_LFS_SKIP_SMUDGE"] = "1"

if not LEROBOT_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", LEROBOT_REPO_URL, str(LEROBOT_DIR)],
        check=True,
        env=env,
    )

subprocess.run(["git", "-C", str(LEROBOT_DIR), "fetch", "--depth", "1", "origin", LEROBOT_REF], check=True, env=env)
subprocess.run(["git", "-C", str(LEROBOT_DIR), "checkout", "FETCH_HEAD"], check=True, env=env)
subprocess.run(["git", "-C", str(LEROBOT_DIR), "rev-parse", "--short", "HEAD"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[training]"], cwd=LEROBOT_DIR, check=True)


In [ ]:
# @title 4. Authenticate Hugging Face and W&B
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_secret(name: str) -> str | None:
    if userdata is not None:
        value = userdata.get(name)
        if value:
            return value
    return os.environ.get(name)

from huggingface_hub import HfApi, get_token, login, notebook_login

hf_token = get_secret("HF_TOKEN") or get_secret("HUGGINGFACE_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=True)
else:
    notebook_login()
    hf_token = get_token()
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token

api = HfApi(token=os.environ.get("HF_TOKEN"))
whoami = api.whoami()
print(f"Hugging Face user: {whoami['name']}")
if whoami["name"] != HF_USER:
    print(f"Configured HF_USER is {HF_USER}; keeping it because dataset and policy names use that value.")

if ENABLE_WANDB:
    import wandb

    wandb_key = get_secret("WANDB_API_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
        wandb.login(key=wandb_key, relogin=True)
    else:
        wandb.login()
    print("W&B login ready.")
else:
    print("W&B disabled for this run.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face user: romando


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: roman-dolgov (roman-dolgov-rd) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login ready.


In [ ]:
# @title 5. Verify the dataset is visible from Colab
info = api.repo_info(DATASET_REPO_ID, repo_type="dataset")
print(f"Dataset found: {info.id}")

files = api.list_repo_files(DATASET_REPO_ID, repo_type="dataset")
print(f"Dataset files: {len(files)} total")
for path in files[:20]:
    print("-", path)


HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'romando/record-test-may15-1930_20260515_194103 '.

In [ ]:
# @title 6. Check CUDA and LeRobot imports
import torch
import lerobot

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"LeRobot import: {lerobot.__file__}")
print(f"Torch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU detected. Training can run on CPU but will be much slower.")


LeRobot import: None
Torch: 2.10.0+cu128
Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
!rm -rf /content/lerobot/outputs/train/

In [ ]:
# @title 7. Train and push the policy
import os
import subprocess
import sys

train_cmd = [
    sys.executable,
    "-m",
    "lerobot.scripts.lerobot_train",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    "--policy.type=lerobot/smolvla_base",
    f"--output_dir={OUTPUT_DIR}",
    f"--job_name={JOB_NAME}",
    f"--policy.device={DEVICE}",
    f"--policy.use_amp={str(USE_AMP and DEVICE == 'cuda').lower()}",
    "--policy.push_to_hub=true",
    f"--policy.repo_id={POLICY_REPO_ID}",
    f"--policy.private={str(POLICY_PRIVATE).lower()}",
    f"--wandb.enable={str(ENABLE_WANDB).lower()}",
    "--wandb.disable_artifact=false",
    f"--batch_size={BATCH_SIZE}",
    f"--num_workers={NUM_WORKERS}",
    f"--steps={TRAIN_STEPS}",
    f"--save_freq={SAVE_FREQ}",
    f"--log_freq={LOG_FREQ}",
    f"--eval_freq={EVAL_FREQ}",
]

if ENABLE_WANDB:
    train_cmd.append(f"--wandb.project={WANDB_PROJECT}")
    if WANDB_ENTITY.strip():
        train_cmd.append(f"--wandb.entity={WANDB_ENTITY.strip()}")

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"
run_env["PYTHONIOENCODING"] = "utf-8"
if os.environ.get("HF_TOKEN"):
    run_env["HF_TOKEN"] = os.environ["HF_TOKEN"]
if os.environ.get("WANDB_API_KEY"):
    run_env["WANDB_API_KEY"] = os.environ["WANDB_API_KEY"]

print("Running:")
print(" ".join(train_cmd), flush=True)

process = subprocess.Popen(
    train_cmd,
    cwd=LEROBOT_DIR,
    env=run_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)

assert process.stdout is not None
for chunk in iter(lambda: process.stdout.read(4096), b""):
    print(chunk.decode("utf-8", errors="replace"), end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, train_cmd)


Running:
/usr/bin/python3 -m lerobot.scripts.lerobot_train --dataset.repo_id=romando/record-test-may16-1200_20260516_124636 --policy.type=act --output_dir=/content/lerobot/outputs/train/job_act-so101-test-may15-2300_20260516_203149 --job_name=job_act-so101-test-may15-2300_20260516_203149 --policy.device=cuda --policy.use_amp=true --policy.push_to_hub=true --policy.repo_id=romando/act-so101-test-may15-2300 --policy.private=true --wandb.enable=true --wandb.disable_artifact=false --batch_size=64 --num_workers=8 --steps=5000 --save_freq=1000 --log_freq=50 --eval_freq=0 --policy.chunk_size=25 --policy.n_action_steps=25 --wandb.project=lerobot --wandb.entity=roman-dolgov-rd
INFO 2026-05-16 20:32:14 ot_train.py:212 {'batch_size': 64,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
        

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# @title 8. Verify the uploaded policy repo
from huggingface_hub import hf_hub_download

files = api.list_repo_files(POLICY_REPO_ID, repo_type="model")
print(f"Policy uploaded to: https://huggingface.co/{POLICY_REPO_ID}")
print(f"Model repo files: {len(files)} total")
for path in files:
    print("-", path)

required = {"config.json", "model.safetensors", "train_config.json"}
missing = sorted(required.difference(files))
if missing:
    raise RuntimeError(f"Upload is missing expected files: {missing}")

config_path = hf_hub_download(
    repo_id=POLICY_REPO_ID,
    filename="train_config.json",
    token=os.environ.get("HF_TOKEN"),
)
print(f"Verified train_config.json download: {config_path}")


Policy uploaded to: https://huggingface.co/romando/act-so101-test-may15-2300
Model repo files: 9 total
- .gitattributes
- README.md
- config.json
- model.safetensors
- policy_postprocessor.json
- policy_postprocessor_step_0_unnormalizer_processor.safetensors
- policy_preprocessor.json
- policy_preprocessor_step_3_normalizer_processor.safetensors
- train_config.json


train_config.json: 0.00B [00:00, ?B/s]

Verified train_config.json download: /root/.cache/huggingface/hub/models--romando--act-so101-test-may15-2300/snapshots/be0d10e880127997d341e637644700d06d7655c4/train_config.json


## Notes

- The `lerobot-train` process pushes the final policy to `POLICY_REPO_ID` because `--policy.push_to_hub=true` and `--policy.repo_id=...` are set.
- W&B model artifacts are enabled with `--wandb.disable_artifact=false`; they are logged at checkpoint save time.
- For a full training run, increase `TRAIN_STEPS` from `1000` to the target duration you want to run on Colab.
